# 95 — Waveform-QC and stack nodal events by Geode metadata

This notebook consumes the authoritative time-window assignments from notebook 94, chooses a consensus waveform reference for each Geode stack, rejects probable non-shot or line-disturbance events, and creates one nodal stack per accepted Geode stack.

Goal:

```text
Geode stack record
    → matched individual nodal events
    → cross-correlate and align accepted nodal events
    → stack traces by station/channel
    → write stacked MiniSEED, SEG-Y, PNG plots, and stack provenance tables
```

Inputs from `lbssp_shot_catalog.sqlite`:

- `geode_nodal_event_matches` from notebook 94
- `geode_nodal_stack_match_summary` from notebook 94
- `shot_gather_files` from notebook 90
- `trace_index` from notebook 90

Outputs/replaces only notebook-95-owned tables:

- `nodal_stacks`
- `nodal_stack_members`
- `nodal_stack_files`
- `nodal_stack_processing_errors`
- `nodal_stack_reference_candidates`
- `nodal_event_catalog_qc`

This notebook does **not** alter 90/92/93/94 tables.

## Repair note

This version fixes a name-shadowing bug in the main processing loop. The local variable
`long_gather_path` was renamed to `member_long_gather_path` so it no longer overwrites
the helper function `long_gather_path(...)` after the first processed stack.

The previous run therefore produced one successful stack and 193 processing errors before
replacing the notebook-95 SQLite tables. Re-run this notebook from the beginning to rebuild
the complete Geode-linked stack catalog.


## 1. Configuration

In [1]:
from pathlib import Path
import sqlite3
import json
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from obspy import read, Stream, Trace, UTCDateTime
from obspy.clients.filesystem.sds import Client as SDSClient
from scipy.signal import correlate, correlation_lags

PROJECT_ROOT = Path("/Volumes/tachyon/LBSSP_DATA")
CATALOG_DB = PROJECT_ROOT / "catalog" / "lbssp_shot_catalog.sqlite"
CATALOG_DB.parent.mkdir(parents=True, exist_ok=True)

OUT_ROOT = PROJECT_ROOT / "95_nodal_stacked_by_geode"
SDS_ROOT = PROJECT_ROOT / "nodal_sds_position_codes"
LONG_GATHER_ROOT = OUT_ROOT / "long_member_gathers"
OUT_ROOT.mkdir(parents=True, exist_ok=True)
LONG_GATHER_ROOT.mkdir(parents=True, exist_ok=True)

TARGET_SURVEYS = None
# TARGET_SURVEYS = ["T1_streamer_masw"]

COMPONENTS_TO_STACK = ["Z"]#,"N", "E"]
PRIMARY_COMPONENT = "Z"

MATCH_STATUSES_TO_USE = ["accepted_auto"]
MIN_MATCH_SCORE = None
MIN_EVENTS_PER_STACK = 2

BANDPASS_FREQMIN_HZ = 5.0
BANDPASS_FREQMAX_HZ = 150.0
XCORR_TMIN_S = 0.0
XCORR_TMAX_S = 0.8
MAX_XCORR_SHIFT_S = 0.45
MIN_CORR_COEF = 0.65
MIN_XCORR_TRACES = 3
MIN_XCORR_OVERLAP_FRACTION = 0.45
MAX_RECEIVER_SHIFT_MAD_S = 0.020
REFERENCE_TOP_N_TRACES = 6
REFERENCE_MAX_CANDIDATES = 8
POSITION_EXPORT_DECIMALS = 2

LONG_GATHER_TMIN_S = -0.50
LONG_GATHER_TMAX_S = 1.50
STACK_TMIN_S = -0.05
STACK_TMAX_S = 1.20

PLOT_TMIN_S = 0.0
PLOT_TMAX_S = 0.8
PLOT_CLIP_PERCENTILE = 99
PLOT_SCALE = 0.8

WRITE_SEGY = True
WRITE_MSEED = True
WRITE_PNG = True
SHOW_PLOTS = False

MAX_STACKS = None

print("CATALOG_DB:", CATALOG_DB)
print("OUT_ROOT:", OUT_ROOT)

CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
OUT_ROOT: /Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_by_geode


## 2. Database safety preflight

In [2]:
if not CATALOG_DB.exists():
    raise FileNotFoundError(
        f"Catalog database not found: {CATALOG_DB}\n"
        "Run 90_*, 92_*, 93_*, and 94_* first."
    )

with sqlite3.connect(CATALOG_DB) as _conn:
    _tables = pd.read_sql(
        """
        SELECT name
        FROM sqlite_master
        WHERE type='table'
        ORDER BY name
        """,
        _conn,
    )["name"].tolist()

REQUIRED_INPUT_TABLES = [
    "shot_events",
    "shot_gather_files",
    "trace_index",
    "geode_events",
    "nodal_source_estimates",
    "geode_nodal_event_matches",
    "geode_nodal_stack_match_summary",
    "nodal_event_catalog",
]

missing = [t for t in REQUIRED_INPUT_TABLES if t not in _tables]
if missing:
    raise RuntimeError(f"Catalog exists, but required input tables are missing: {missing}")

NOTEBOOK_95_TABLES = [
    "nodal_stacks",
    "nodal_stack_members",
    "nodal_stack_files",
    "nodal_stack_processing_errors",
    "nodal_stack_reference_candidates",
    "nodal_event_catalog_qc",
]

print("Input catalog checked:", CATALOG_DB)
print("Required input tables present.")
print("95 will only create/replace:", NOTEBOOK_95_TABLES)

Input catalog checked: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
Required input tables present.
95 will only create/replace: ['nodal_stacks', 'nodal_stack_members', 'nodal_stack_files', 'nodal_stack_processing_errors', 'nodal_stack_reference_candidates', 'nodal_event_catalog_qc']


## 3. Load input tables

In [3]:
conn = sqlite3.connect(CATALOG_DB)

matches = pd.read_sql("SELECT * FROM geode_nodal_event_matches", conn)
stack_match_summary = pd.read_sql("SELECT * FROM geode_nodal_stack_match_summary", conn)
shot_gather_files = pd.read_sql("SELECT * FROM shot_gather_files WHERE instrument_system='nodal'", conn)
trace_index = pd.read_sql("SELECT * FROM trace_index WHERE instrument_system='nodal'", conn)
event_catalog_base = pd.read_sql("SELECT * FROM nodal_event_catalog", conn)

print("matches:", len(matches))
print("stack_match_summary:", len(stack_match_summary))
print("shot_gather_files:", len(shot_gather_files))
print("trace_index:", len(trace_index))
print("nodal_event_catalog:", len(event_catalog_base))

matches["accepted"] = matches["accepted"].astype(str).str.lower().isin(["true", "1", "yes"])
matches["nodal_event_time_dt"] = pd.to_datetime(matches["nodal_event_time_utc"], errors="coerce", utc=True)
matches["geode_final_trigger_dt"] = pd.to_datetime(matches["geode_final_trigger_time_utc"], errors="coerce", utc=True)

for col in ["source_x_truth_m", "estimated_source_x_m", "source_x_residual_m", "match_score", "time_from_final_trigger_s"]:
    matches[col] = pd.to_numeric(matches[col], errors="coerce")

display(matches.groupby(["geode_survey", "match_status"], dropna=False).size().reset_index(name="n"))

matches: 966
stack_match_summary: 199
shot_gather_files: 3436
trace_index: 338580
nodal_event_catalog: 3436


,geode_survey,match_status,n
0,T1_1m_refraction,accepted_auto,252
1,T1_2m_refraction,accepted_auto,225
2,T1_streamer_masw,accepted_auto,296
3,T3_1m_refraction,accepted_auto,193


## 4. Select stacks to process

In [4]:
use = matches.copy()

if TARGET_SURVEYS is not None:
    use = use[use["geode_survey"].astype(str).isin(TARGET_SURVEYS)].copy()

use = use[use["match_status"].astype(str).isin(MATCH_STATUSES_TO_USE)].copy()

if MIN_MATCH_SCORE is not None:
    use = use[use["match_score"] >= MIN_MATCH_SCORE].copy()

use = use.dropna(subset=["geode_event_id", "nodal_event_id", "nodal_event_time_dt", "source_x_truth_m"]).copy()

def resolve_mseed_path(row):
    catalog_path = Path(str(row.get("mseed_path", "")))
    candidates = [catalog_path]
    label = str(row.get("nodal_timewindow_label", ""))
    if catalog_path.name and label:
        candidates.extend([
            PROJECT_ROOT / "nodal_unstacked_shotgathers" / label / "gathers_mseed" / catalog_path.name,
            PROJECT_ROOT / "nodal_fullnode_shotgathers_v4" / label / "gathers_mseed" / catalog_path.name,
        ])
    for candidate in candidates:
        if candidate.exists():
            return str(candidate)
    return None

use["mseed_path_catalog"] = use.get("mseed_path")
use["mseed_path"] = use.apply(resolve_mseed_path, axis=1)
use["mseed_path_exists"] = use["mseed_path"].notna()
use = use[use["mseed_path_exists"]].copy()

group_counts = (
    use.groupby(["geode_event_id", "geode_survey", "line", "file_no", "source_x_truth_m"], dropna=False)
    .size()
    .reset_index(name="n_candidate_members")
    .sort_values(["geode_survey", "file_no", "source_x_truth_m"])
)

group_counts = group_counts[group_counts["n_candidate_members"] >= MIN_EVENTS_PER_STACK].copy()

if MAX_STACKS is not None:
    group_counts = group_counts.head(MAX_STACKS).copy()

print("Candidate matched events:", len(use))
print("Stacks to process:", len(group_counts))
display(group_counts.head(100))

Candidate matched events: 966
Stacks to process: 194


,geode_event_id,geode_survey,line,file_no,source_x_truth_m,n_candidate_members
0,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,6
1,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,3
2,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,7
3,GEODE_T1_1M_REFRACTION_F3010,T1_1m_refraction,T1,3010,92.5,2
4,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,6
...,...,...,...,...,...,...
99,GEODE_T1_STREAMER_MASW_F1023,T1_streamer_masw,T1,1023,120.0,3
100,GEODE_T1_STREAMER_MASW_F1024,T1_streamer_masw,T1,1024,121.5,2
101,GEODE_T1_STREAMER_MASW_F1025,T1_streamer_masw,T1,1025,123.0,3
102,GEODE_T1_STREAMER_MASW_F1026,T1_streamer_masw,T1,1026,124.5,4


## 5. Geometry and waveform helpers

In [5]:
def safe_name(s):
    s = str(s)
    for ch in [" ", "/", "\\", ":", ";", ",", "(", ")", "[", "]"]:
        s = s.replace(ch, "_")
    return s


def attach_receiver_x(st, event_id, component=None):
    geom = trace_index[trace_index["event_id"].astype(str) == str(event_id)].copy()
    if component is not None:
        geom = geom[geom["channel"].astype(str).str.endswith(component)].copy()

    geom["receiver_x_m"] = pd.to_numeric(geom["receiver_x_m"], errors="coerce")
    geom = geom.dropna(subset=["receiver_x_m"])

    lookup = {}
    for _, r in geom.iterrows():
        station = str(r.get("station", ""))
        channel = str(r.get("channel", ""))
        network = str(r.get("network", ""))
        location = "" if pd.isna(r.get("location", "")) else str(r.get("location", ""))

        lookup[(station, channel)] = float(r["receiver_x_m"])
        lookup[f"{network}.{station}.{location}.{channel}"] = float(r["receiver_x_m"])
        if "seed_id" in geom.columns and pd.notna(r.get("seed_id")):
            lookup[str(r["seed_id"])] = float(r["receiver_x_m"])

    out = st.copy()
    for tr in out:
        x = lookup.get((tr.stats.station, tr.stats.channel))
        if x is None:
            x = lookup.get(tr.id)
        if x is not None:
            tr.stats.receiver_x_m = float(x)

    return out


def preprocess_stream(st, component):
    stc = st.select(channel=f"*{component}").copy()
    for tr in stc:
        tr.data = tr.data.astype(np.float64)
        tr.detrend("linear")
        tr.taper(max_percentage=0.02)
        nyq = 0.5 / tr.stats.delta
        if BANDPASS_FREQMAX_HZ < 0.95 * nyq:
            tr.filter("bandpass", freqmin=BANDPASS_FREQMIN_HZ, freqmax=BANDPASS_FREQMAX_HZ, corners=4, zerophase=True)
        else:
            tr.filter("highpass", freq=BANDPASS_FREQMIN_HZ, corners=4, zerophase=True)
    return stc


def stream_time_origin(row):
    return UTCDateTime(pd.Timestamp(row["nodal_event_time_dt"]).to_pydatetime())


def trace_relative_series(tr, origin, tmin, tmax, dt):
    t_grid = np.arange(tmin, tmax + 0.5 * dt, dt, dtype=float)
    tr_t0 = tr.stats.starttime - origin
    tr_t = tr_t0 + np.arange(tr.stats.npts, dtype=float) * tr.stats.delta
    y = np.interp(t_grid, tr_t, tr.data.astype(float), left=np.nan, right=np.nan)
    return t_grid, y


def normalize_for_xcorr(y):
    y = np.asarray(y, dtype=float)
    good = np.isfinite(y)
    if good.sum() < 10:
        return None
    z = y - np.nanmedian(y)
    std = np.nanstd(z)
    if not np.isfinite(std) or std <= 0:
        return None
    z[~good] = np.nan
    return z / std


def xcorr_shift_seconds(reference, candidate, dt, max_shift_s=MAX_XCORR_SHIFT_S):
    ref_values = normalize_for_xcorr(reference)
    cand_values = normalize_for_xcorr(candidate)
    if ref_values is None or cand_values is None:
        return np.nan, np.nan, 0
    n_total = min(len(ref_values), len(cand_values))
    ref_values = ref_values[:n_total]
    cand_values = cand_values[:n_total]
    min_overlap = max(20, int(np.ceil(MIN_XCORR_OVERLAP_FRACTION * n_total)))

    ref_mask = np.isfinite(ref_values).astype(float)
    cand_mask = np.isfinite(cand_values).astype(float)
    ref = np.where(np.isfinite(ref_values), ref_values, 0.0)
    cand = np.where(np.isfinite(cand_values), cand_values, 0.0)

    lags = correlation_lags(n_total, n_total, mode="full")
    counts = correlate(cand_mask, ref_mask, mode="full", method="auto")
    dot = correlate(cand, ref, mode="full", method="auto")
    sum_cand = correlate(cand, ref_mask, mode="full", method="auto")
    sum_ref = correlate(cand_mask, ref, mode="full", method="auto")
    sumsq_cand = correlate(cand ** 2, ref_mask, mode="full", method="auto")
    sumsq_ref = correlate(cand_mask, ref ** 2, mode="full", method="auto")

    allowed = (
        (np.abs(lags) <= int(round(max_shift_s / dt)))
        & (counts >= min_overlap)
    )
    coefficients = np.full(dot.shape, np.nan, dtype=float)
    good = allowed & (counts > 0)
    covariance = dot[good] - sum_cand[good] * sum_ref[good] / counts[good]
    variance_cand = sumsq_cand[good] - sum_cand[good] ** 2 / counts[good]
    variance_ref = sumsq_ref[good] - sum_ref[good] ** 2 / counts[good]
    denominator = np.sqrt(np.maximum(variance_cand, 0) * np.maximum(variance_ref, 0))
    valid = denominator > 0
    good_indices = np.flatnonzero(good)
    coefficients[good_indices[valid]] = covariance[valid] / denominator[valid]
    if not np.isfinite(coefficients).any():
        return np.nan, np.nan, 0
    position = int(np.nanargmax(coefficients))
    return (
        float(-lags[position] * dt),
        float(coefficients[position]),
        int(round(counts[position])),
    )

def trace_key(tr):
    return (str(tr.stats.station), str(tr.stats.channel))


def stream_by_key(st):
    return {trace_key(tr): tr for tr in st}

sds = SDSClient(str(SDS_ROOT))


def long_gather_path(event_id):
    return LONG_GATHER_ROOT / f"{safe_name(event_id)}_DPall_long.mseed"


def seed_pairs_from_catalog_gather(row):
    catalog_stream = read(str(row["mseed_path"]), headonly=True)
    pairs = sorted({
        (str(tr.stats.network), str(tr.stats.location))
        for tr in catalog_stream
        if str(tr.stats.network)
    })
    if not pairs:
        raise RuntimeError(f"Cannot infer network/location for {row['nodal_event_id']}")
    return pairs


def load_long_member_gathers(members):
    """Load one continuous SDS block, then slice every member in the stack group."""
    results = {}
    missing_rows = []
    pair_lookup = {}

    for _, row in members.iterrows():
        event_id = str(row["nodal_event_id"])
        out_path = long_gather_path(event_id)
        if out_path.exists():
            results[event_id] = (read(str(out_path)), str(out_path))
        else:
            pairs = seed_pairs_from_catalog_gather(row)
            pair_lookup[event_id] = pairs
            missing_rows.append(row)

    if not missing_rows:
        return results

    missing = pd.DataFrame(missing_rows)
    origins = {
        str(row["nodal_event_id"]): stream_time_origin(row)
        for _, row in missing.iterrows()
    }
    all_pairs = sorted({
        pair for pairs in pair_lookup.values() for pair in pairs
    })
    continuous_by_pair = {}
    for network, location in all_pairs:
        relevant_ids = [
            event_id for event_id, pairs in pair_lookup.items()
            if (network, location) in pairs
        ]
        start = min(origins[event_id] for event_id in relevant_ids) + LONG_GATHER_TMIN_S - 0.10
        end = max(origins[event_id] for event_id in relevant_ids) + LONG_GATHER_TMAX_S + 0.10
        continuous = sds.get_waveforms(
            network, "*", location, "DP?", start, end
        )
        continuous.merge(method=1, fill_value="interpolate")
        if not continuous:
            raise RuntimeError(
                f"No continuous SDS block for {network}.*.{location}.DP? {start} to {end}"
            )
        continuous_by_pair[(network, location)] = continuous

    for _, row in missing.iterrows():
        event_id = str(row["nodal_event_id"])
        origin = origins[event_id]
        gather = Stream()
        for pair in pair_lookup[event_id]:
            gather += continuous_by_pair[pair].slice(
                origin + LONG_GATHER_TMIN_S,
                origin + LONG_GATHER_TMAX_S,
            ).copy()
        if not gather:
            raise RuntimeError(f"Empty long continuous-SDS gather for {event_id}")
        out_path = long_gather_path(event_id)
        gather.write(str(out_path), format="MSEED")
        results[event_id] = (gather, str(out_path))

    return results


## 6. Reference trace selection and event-level shift estimation

In [6]:
def select_reference_trace_keys(ref_st, ref_origin, component=PRIMARY_COMPONENT):
    rows = []
    for tr in ref_st.select(channel=f"*{component}"):
        if not hasattr(tr.stats, "receiver_x_m"):
            continue
        dt = float(tr.stats.delta)
        _, y = trace_relative_series(tr, ref_origin, XCORR_TMIN_S, XCORR_TMAX_S, dt)
        if np.isfinite(y).sum() < 10:
            continue
        y0 = y - np.nanmedian(y)
        e = float(np.nansum(y0**2))
        rows.append({
            "key": trace_key(tr),
            "station": tr.stats.station,
            "channel": tr.stats.channel,
            "receiver_x_m": float(tr.stats.receiver_x_m),
            "energy": e,
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return [], df

    df = df.sort_values("energy", ascending=False).head(REFERENCE_TOP_N_TRACES).copy()
    return list(df["key"]), df


def estimate_member_shift(ref_st, ref_origin, cand_st, cand_origin, ref_keys):
    ref_map = stream_by_key(ref_st)
    cand_map = stream_by_key(cand_st)

    shifts = []
    corrs = []
    overlap_counts = []

    for k in ref_keys:
        if k not in ref_map or k not in cand_map:
            continue

        rtr = ref_map[k]
        ctr = cand_map[k]

        dt = float(rtr.stats.delta)
        if abs(float(ctr.stats.delta) - dt) > 1e-6:
            continue

        _, yref = trace_relative_series(rtr, ref_origin, XCORR_TMIN_S, XCORR_TMAX_S, dt)
        _, ycand = trace_relative_series(ctr, cand_origin, XCORR_TMIN_S, XCORR_TMAX_S, dt)

        shift, corr, n_overlap = xcorr_shift_seconds(
            yref, ycand, dt, MAX_XCORR_SHIFT_S
        )
        if np.isfinite(shift) and np.isfinite(corr):
            shifts.append(shift)
            corrs.append(corr)
            overlap_counts.append(n_overlap)

    if not shifts:
        return np.nan, np.nan, 0, np.nan, []

    median_shift = float(np.nanmedian(shifts))
    shift_mad = float(1.4826 * np.nanmedian(np.abs(np.asarray(shifts) - median_shift)))
    return median_shift, float(np.nanmedian(corrs)), int(len(shifts)), shift_mad, [
        {"shift_s": float(s), "corrcoef": float(c), "n_overlap_samples": int(n)}
        for s, c, n in zip(shifts, corrs, overlap_counts)
    ]


def select_consensus_reference(members, member_streams, member_origins):
    candidate_rows = (
        members.assign(abs_time_from_final=members["time_from_final_trigger_s"].abs())
        .sort_values(["match_score", "abs_time_from_final"], ascending=[False, True])
        .head(REFERENCE_MAX_CANDIDATES)
    )

    diagnostics = []
    for _, candidate_row in candidate_rows.iterrows():
        candidate_id = candidate_row["nodal_event_id"]
        candidate_stream = member_streams[candidate_id]
        candidate_origin = member_origins[candidate_id]
        reference_keys, _ = select_reference_trace_keys(
            candidate_stream, candidate_origin, component=PRIMARY_COMPONENT
        )

        correlations = []
        supported_members = 0
        for member_id, member_stream in member_streams.items():
            if member_id == candidate_id:
                continue
            shift_s, corrcoef, n_corr, shift_mad, _ = estimate_member_shift(
                candidate_stream, candidate_origin, member_stream,
                member_origins[member_id], reference_keys,
            )
            if np.isfinite(corrcoef) and n_corr >= MIN_XCORR_TRACES:
                correlations.append(corrcoef)
                if (
                    corrcoef >= MIN_CORR_COEF
                    and np.isfinite(shift_s)
                    and abs(shift_s) <= MAX_XCORR_SHIFT_S
                    and np.isfinite(shift_mad)
                    and shift_mad <= MAX_RECEIVER_SHIFT_MAD_S
                ):
                    supported_members += 1

        median_corr = float(np.nanmedian(correlations)) if correlations else np.nan
        diagnostics.append({
            "reference_nodal_event_id": candidate_id,
            "n_reference_traces": len(reference_keys),
            "n_other_members_tested": len(member_streams) - 1,
            "n_supported_other_members": supported_members,
            "median_corr_to_other_members": median_corr,
            "original_match_score": candidate_row.get("match_score"),
            "abs_time_from_final_s": candidate_row.get("abs_time_from_final"),
        })

    diagnostic_df = pd.DataFrame(diagnostics)
    if diagnostic_df.empty:
        raise RuntimeError("No usable reference candidates")

    diagnostic_df["median_corr_sort"] = diagnostic_df["median_corr_to_other_members"].fillna(-np.inf)
    diagnostic_df = diagnostic_df.sort_values(
        ["n_supported_other_members", "median_corr_sort", "original_match_score", "abs_time_from_final_s"],
        ascending=[False, False, False, True],
    ).reset_index(drop=True)
    diagnostic_df["selected_reference"] = False
    diagnostic_df.loc[0, "selected_reference"] = True
    selected_id = diagnostic_df.loc[0, "reference_nodal_event_id"]
    return selected_id, diagnostic_df.drop(columns=["median_corr_sort"])

## 7. Stacking and plotting helpers

In [7]:
def build_stacked_stream(member_streams, member_origins, member_shifts, component):
    dt_values = []
    for st in member_streams.values():
        for tr in st.select(channel=f"*{component}"):
            dt_values.append(float(tr.stats.delta))
    if not dt_values:
        return Stream()

    dt = float(np.median(dt_values))
    t_grid = np.arange(STACK_TMIN_S, STACK_TMAX_S + 0.5 * dt, dt, dtype=float)

    all_keys = sorted(set(
        trace_key(tr)
        for st in member_streams.values()
        for tr in st.select(channel=f"*{component}")
        if hasattr(tr.stats, "receiver_x_m")
    ))

    out = Stream()

    for k in all_keys:
        arrays = []
        receiver_x = None
        template = None

        for event_id, st in member_streams.items():
            mp = stream_by_key(st)
            if k not in mp:
                continue

            tr = mp[k]
            if receiver_x is None and hasattr(tr.stats, "receiver_x_m"):
                receiver_x = float(tr.stats.receiver_x_m)
            if template is None:
                template = tr

            origin = member_origins[event_id]
            shift = float(member_shifts.get(event_id, 0.0))

            tr_t0 = tr.stats.starttime - origin
            tr_t = tr_t0 + np.arange(tr.stats.npts, dtype=float) * tr.stats.delta
            y = np.interp(t_grid - shift, tr_t, tr.data.astype(float), left=np.nan, right=np.nan)
            arrays.append(y)

        if not arrays or template is None:
            continue

        A = np.vstack(arrays)
        valid_counts = np.sum(np.isfinite(A), axis=0)
        ystack = np.divide(
            np.nansum(A, axis=0), valid_counts,
            out=np.zeros(A.shape[1], dtype=float), where=valid_counts > 0,
        )

        stats = template.stats.copy()
        stats.starttime = UTCDateTime(0) + STACK_TMIN_S
        stats.delta = dt
        stats.npts = len(ystack)
        stats.processing = list(getattr(stats, "processing", [])) + [f"stacked {len(arrays)} nodal events by notebook 95"]
        stats.receiver_x_m = receiver_x if receiver_x is not None else np.nan
        stats.pop("mseed", None)

        out += Trace(data=ystack.astype(np.float32), header=stats)

    return out


def plot_wiggle_stream(st, title, source_x_m=None, out_png=None, tmin=PLOT_TMIN_S, tmax=PLOT_TMAX_S):
    traces = [tr for tr in st if hasattr(tr.stats, "receiver_x_m")]
    if not traces:
        print("No traces with receiver_x_m to plot:", title)
        return

    traces = sorted(traces, key=lambda tr: float(tr.stats.receiver_x_m))
    dt = float(np.median([tr.stats.delta for tr in traces]))
    t_grid = np.arange(tmin, tmax + 0.5*dt, dt)

    X = []
    Y = []
    for tr in traces:
        x = float(tr.stats.receiver_x_m)
        tr_t0 = tr.stats.starttime - UTCDateTime(0)
        tr_t = tr_t0 + np.arange(tr.stats.npts) * tr.stats.delta
        y = np.interp(t_grid, tr_t, tr.data.astype(float), left=np.nan, right=np.nan)
        if np.isfinite(y).sum() < 5:
            continue
        y = y - np.nanmedian(y)
        X.append(x)
        Y.append(y)

    if not Y:
        print("No valid waveform arrays to plot:", title)
        return

    X = np.asarray(X, dtype=float)
    A = np.vstack(Y)
    clip = np.nanpercentile(np.abs(A), PLOT_CLIP_PERCENTILE)
    if not np.isfinite(clip) or clip <= 0:
        clip = np.nanmax(np.abs(A))
    if not np.isfinite(clip) or clip <= 0:
        clip = 1.0

    xs = np.sort(np.unique(X))
    dx = np.nanmedian(np.diff(xs)) if len(xs) > 1 else 1.0
    if not np.isfinite(dx) or dx <= 0:
        dx = 1.0

    fig, ax = plt.subplots(figsize=(12, 6))
    for x, y in zip(X, A):
        yy = np.clip(y / clip, -1, 1) * dx * PLOT_SCALE
        ax.plot(x + yy, t_grid, linewidth=0.7)
        ax.fill_betweenx(t_grid, x, x + np.maximum(yy, 0), alpha=0.25)

    if source_x_m is not None and np.isfinite(source_x_m):
        ax.axvline(float(source_x_m), linestyle="--", linewidth=1.2, label=f"source x={source_x_m:.1f} m")
        ax.legend(loc="best")

    ax.invert_yaxis()
    ax.set_xlabel("Receiver x (m)")
    ax.set_ylabel("Time relative to stack origin (s)")
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()

    if out_png is not None:
        out_png = Path(out_png)
        out_png.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(out_png, dpi=180)
    if SHOW_PLOTS:
        plt.show()
    else:
        plt.close(fig)


def try_write_segy(st, out_sgy, source_x_m):
    try:
        from obspy.io.segy.segy import SEGYTraceHeader
        from obspy.core import AttribDict

        st2 = st.copy()
        for i, tr in enumerate(st2, start=1):
            if not hasattr(tr.stats, "segy"):
                tr.stats.segy = AttribDict()
            tr.stats.segy.trace_header = SEGYTraceHeader()
            h = tr.stats.segy.trace_header

            rx = float(getattr(tr.stats, "receiver_x_m", np.nan))
            sx = float(source_x_m) if source_x_m is not None and np.isfinite(source_x_m) else np.nan

            h.trace_sequence_number_within_line = i
            h.trace_sequence_number_within_segy_file = i
            h.coordinate_scalar = -100
            if np.isfinite(sx):
                h.source_coordinate_x = int(round(sx * 100))
            if np.isfinite(rx):
                h.group_coordinate_x = int(round(rx * 100))
                if np.isfinite(sx):
                    h.distance_from_center_of_the_source_point_to_the_center_of_the_receiver_group = int(round((rx - sx) * 100))

        out_sgy = Path(out_sgy)
        out_sgy.parent.mkdir(parents=True, exist_ok=True)
        st2.write(str(out_sgy), format="SEGY", data_encoding=5)
        return True, None
    except Exception as e:
        return False, repr(e)

## 8. Process stacks

In [8]:
stack_rows = []
member_rows_out = []
reference_candidate_rows = []
file_rows = []
error_rows = []

for ii, g in group_counts.reset_index(drop=True).iterrows():
    geode_event_id = g["geode_event_id"]
    print(f"[{ii+1}/{len(group_counts)}] {geode_event_id}")

    members = use[use["geode_event_id"].astype(str).eq(str(geode_event_id))].copy()
    members = members.sort_values(["time_from_final_trigger_s", "match_score"], ascending=[True, False]).reset_index(drop=True)

    survey = str(g["geode_survey"])
    line = str(g["line"])
    file_no = int(g["file_no"]) if pd.notna(g["file_no"]) and np.isfinite(g["file_no"]) else -1
    source_x_truth_m = float(g["source_x_truth_m"])

    stack_id = f"NODALSTACK_{safe_name(line)}_{safe_name(survey)}_F{file_no:04d}_x{source_x_truth_m:06.1f}m"
    stack_dir = OUT_ROOT / safe_name(line) / safe_name(survey) / stack_id
    stack_dir.mkdir(parents=True, exist_ok=True)

    try:
        member_streams = {}
        member_origins = {}

        long_member_gathers = load_long_member_gathers(members)

        for _, row in members.iterrows():
            event_id = row["nodal_event_id"]
            st, member_long_gather_path = long_member_gathers[event_id]
            st = attach_receiver_x(st, event_id, component=None)
            origin = stream_time_origin(row)

            st_proc = Stream()
            for comp in COMPONENTS_TO_STACK:
                st_proc += preprocess_stream(st, comp)

            member_streams[event_id] = st_proc
            member_origins[event_id] = origin

        ref_event_id, reference_diagnostics = select_consensus_reference(
            members, member_streams, member_origins
        )
        ref_row = members[members["nodal_event_id"].astype(str).eq(str(ref_event_id))].iloc[0]
        ref_origin = member_origins[ref_event_id]
        for _, reference_diagnostic in reference_diagnostics.iterrows():
            diagnostic_record = reference_diagnostic.to_dict()
            diagnostic_record.update({
                "stack_id": stack_id,
                "geode_event_id": geode_event_id,
                "geode_survey": survey,
                "line": line,
                "file_no": file_no,
                "source_x_truth_m": source_x_truth_m,
            })
            reference_candidate_rows.append(diagnostic_record)

        ref_st = member_streams[ref_event_id]
        ref_keys, ref_key_df = select_reference_trace_keys(ref_st, ref_origin, component=PRIMARY_COMPONENT)

        if not ref_keys:
            raise RuntimeError("No reference trace keys found for cross-correlation")

        member_shifts = {}
        member_corrs = {}
        member_ncorr = {}
        member_trace_corr_json = {}
        member_shift_mads = {}
        member_long_gather_paths = {}

        for _, row in members.iterrows():
            event_id = row["nodal_event_id"]
            st = member_streams[event_id]
            origin = member_origins[event_id]

            if event_id == ref_event_id:
                shift_s = 0.0
                corrcoef = 1.0
                n_corr = len(ref_keys)
                shift_mad = 0.0
                corr_detail = [
                    {"shift_s": 0.0, "corrcoef": 1.0, "n_overlap_samples": None}
                    for _ in ref_keys
                ]
            else:
                shift_s, corrcoef, n_corr, shift_mad, corr_detail = estimate_member_shift(
                    ref_st, ref_origin, st, origin, ref_keys
                )

            member_shifts[event_id] = shift_s
            member_corrs[event_id] = corrcoef
            member_ncorr[event_id] = n_corr
            member_shift_mads[event_id] = shift_mad
            member_long_gather_paths[event_id] = str(
                LONG_GATHER_ROOT / f"{safe_name(event_id)}_DPall_long.mseed"
            )
            member_trace_corr_json[event_id] = json.dumps(corr_detail)

        accepted_member_ids = []
        for _, row in members.iterrows():
            event_id = row["nodal_event_id"]
            corrcoef = member_corrs.get(event_id, np.nan)
            shift_s = member_shifts.get(event_id, np.nan)
            n_corr = member_ncorr.get(event_id, 0)
            shift_mad = member_shift_mads.get(event_id, np.nan)

            accepted_for_stack = (
                np.isfinite(corrcoef)
                and corrcoef >= MIN_CORR_COEF
                and np.isfinite(shift_s)
                and abs(shift_s) <= MAX_XCORR_SHIFT_S
                and n_corr >= MIN_XCORR_TRACES
                and np.isfinite(shift_mad)
                and shift_mad <= MAX_RECEIVER_SHIFT_MAD_S
            )

            if event_id == ref_event_id:
                accepted_for_stack = True
                rejection_reason = "accepted_consensus_reference"
            elif n_corr < MIN_XCORR_TRACES:
                rejection_reason = "rejected_insufficient_common_traces"
            elif not np.isfinite(corrcoef):
                rejection_reason = "rejected_invalid_correlation"
            elif corrcoef < MIN_CORR_COEF:
                rejection_reason = "rejected_low_correlation_probable_nonshot"
            elif not np.isfinite(shift_s):
                rejection_reason = "rejected_invalid_shift"
            elif abs(shift_s) > MAX_XCORR_SHIFT_S:
                rejection_reason = "rejected_shift_exceeds_limit"
            elif not np.isfinite(shift_mad) or shift_mad > MAX_RECEIVER_SHIFT_MAD_S:
                rejection_reason = "rejected_inconsistent_receiver_shifts"
            else:
                rejection_reason = "accepted_waveform"

            if accepted_for_stack:
                accepted_member_ids.append(event_id)

            member_rows_out.append({
                "stack_id": stack_id,
                "geode_event_id": geode_event_id,
                "geode_survey": survey,
                "line": line,
                "file_no": file_no,
                "source_x_truth_m": source_x_truth_m,
                "nodal_event_id": event_id,
                "nodal_event_time_utc": row["nodal_event_time_utc"],
                "time_from_final_trigger_s": row["time_from_final_trigger_s"],
                "estimated_source_x_m": row["estimated_source_x_m"],
                "source_x_residual_m": row["source_x_residual_m"],
                "match_status": row["match_status"],
                "match_score": row["match_score"],
                "is_reference_event": event_id == ref_event_id,
                "xcorr_shift_s": shift_s,
                "xcorr_corrcoef": corrcoef,
                "xcorr_n_traces": n_corr,
                "xcorr_shift_mad_s": shift_mad,
                "corrected_nodal_event_time_utc": (
                    pd.Timestamp(row["nodal_event_time_dt"])
                    - pd.to_timedelta(shift_s, unit="s")
                    if np.isfinite(shift_s) else pd.NaT
                ),
                "accepted_for_stack": accepted_for_stack,
                "waveform_qc_status": "accepted_waveform" if accepted_for_stack else "rejected_waveform",
                "rejection_reason": rejection_reason,
                "final_event_status": "accepted_for_stack" if accepted_for_stack else "rejected_probable_nonshot_or_line_disturbance",
                "trace_corr_json": member_trace_corr_json.get(event_id),
                "mseed_path": row["mseed_path"],
                "long_mseed_path": member_long_gather_paths.get(event_id),
            })

        if len(accepted_member_ids) < MIN_EVENTS_PER_STACK:
            raise RuntimeError(f"Only {len(accepted_member_ids)} accepted members after xcorr")

        accepted_streams = {eid: member_streams[eid] for eid in accepted_member_ids}
        accepted_origins = {eid: member_origins[eid] for eid in accepted_member_ids}
        accepted_shifts = {eid: member_shifts[eid] for eid in accepted_member_ids}

        for comp in COMPONENTS_TO_STACK:
            st_stack = build_stacked_stream(accepted_streams, accepted_origins, accepted_shifts, component=comp)
            if len(st_stack) == 0:
                continue

            comp_dir = stack_dir / comp
            comp_dir.mkdir(parents=True, exist_ok=True)

            base = f"{stack_id}_{comp}_{len(accepted_member_ids):02d}ev"

            if WRITE_MSEED:
                out_mseed = comp_dir / f"{base}.mseed"
                st_stack.write(str(out_mseed), format="MSEED")
                file_rows.append({
                    "stack_id": stack_id,
                    "geode_event_id": geode_event_id,
                    "component": comp,
                    "file_type": "mseed",
                    "file_path": str(out_mseed),
                    "n_traces": len(st_stack),
                    "n_stack_members": len(accepted_member_ids),
                })

            if WRITE_SEGY:
                out_sgy = comp_dir / f"{base}.sgy"
                ok, err = try_write_segy(st_stack, out_sgy, source_x_truth_m)
                if ok:
                    file_rows.append({
                        "stack_id": stack_id,
                        "geode_event_id": geode_event_id,
                        "component": comp,
                        "file_type": "segy",
                        "file_path": str(out_sgy),
                        "n_traces": len(st_stack),
                        "n_stack_members": len(accepted_member_ids),
                    })
                else:
                    error_rows.append({
                        "stack_id": stack_id,
                        "geode_event_id": geode_event_id,
                        "stage": "write_segy",
                        "error": err,
                    })

            if WRITE_PNG:
                out_png = comp_dir / f"{base}_wiggle.png"
                title = (
                    f"{line} {survey} Geode F{file_no} x={source_x_truth_m:.1f} m\n"
                    f"Nodal stack: {len(accepted_member_ids)} events, component {comp}"
                )
                plot_wiggle_stream(st_stack, title=title, source_x_m=source_x_truth_m, out_png=out_png)
                file_rows.append({
                    "stack_id": stack_id,
                    "geode_event_id": geode_event_id,
                    "component": comp,
                    "file_type": "png_wiggle",
                    "file_path": str(out_png),
                    "n_traces": len(st_stack),
                    "n_stack_members": len(accepted_member_ids),
                })

        accepted_times = pd.to_numeric(
            members[members["nodal_event_id"].isin(accepted_member_ids)]["time_from_final_trigger_s"],
            errors="coerce",
        )

        stack_rows.append({
            "stack_id": stack_id,
            "geode_event_id": geode_event_id,
            "geode_survey": survey,
            "line": line,
            "file_no": file_no,
            "source_x_truth_m": source_x_truth_m,
            "source_type": members["source_type"].dropna().iloc[0] if "source_type" in members.columns and members["source_type"].notna().any() else None,
            "n_candidate_members": len(members),
            "n_accepted_members": len(accepted_member_ids),
            "n_rejected_members": len(members) - len(accepted_member_ids),
            "reference_nodal_event_id": ref_event_id,
            "reference_supported_other_members": int(reference_diagnostics.loc[0, "n_supported_other_members"]),
            "reference_median_corr_to_other_members": float(reference_diagnostics.loc[0, "median_corr_to_other_members"]) if pd.notna(reference_diagnostics.loc[0, "median_corr_to_other_members"]) else np.nan,
            "median_xcorr_shift_s": float(np.nanmedian([member_shifts[eid] for eid in accepted_member_ids])),
            "median_xcorr_corrcoef": float(np.nanmedian([member_corrs[eid] for eid in accepted_member_ids])),
            "min_member_time_from_final_s": float(accepted_times.min()),
            "max_member_time_from_final_s": float(accepted_times.max()),
            "output_dir": str(stack_dir),
            "status": "ok",
        })

    except Exception as e:
        print("  ERROR:", e)
        error_rows.append({
            "stack_id": locals().get("stack_id", None),
            "geode_event_id": geode_event_id,
            "stage": "process_stack",
            "error": repr(e),
            "traceback": traceback.format_exc(),
        })

nodal_stacks = pd.DataFrame(stack_rows)
nodal_stack_members = pd.DataFrame(member_rows_out)
nodal_stack_reference_candidates = pd.DataFrame(reference_candidate_rows)
nodal_stack_files = pd.DataFrame(file_rows)
nodal_stack_processing_errors = pd.DataFrame(error_rows)

successful_stack_ids = set(nodal_stacks.get("stack_id", pd.Series(dtype=str)).dropna().astype(str))
if len(nodal_stack_members):
    nodal_stack_members["stack_output_status"] = np.where(
        nodal_stack_members["stack_id"].astype(str).isin(successful_stack_ids),
        "stack_written", "stack_failed",
    )
    nodal_stack_members["included_in_output_stack"] = (
        nodal_stack_members["accepted_for_stack"].astype(bool)
        & nodal_stack_members["stack_id"].astype(str).isin(successful_stack_ids)
    )
    failed_after_waveform = (
        nodal_stack_members["accepted_for_stack"].astype(bool)
        & ~nodal_stack_members["included_in_output_stack"]
    )
    nodal_stack_members.loc[failed_after_waveform, "final_event_status"] = "not_stacked_processing_failure"

event_catalog_qc = event_catalog_base.drop(
    columns=["waveform_qc_status", "final_event_status"], errors="ignore"
).copy()
member_qc_columns = [
    "nodal_event_id", "stack_id", "is_reference_event",
    "xcorr_shift_s", "xcorr_corrcoef", "xcorr_n_traces", "xcorr_shift_mad_s",
    "corrected_nodal_event_time_utc",
    "accepted_for_stack", "included_in_output_stack", "stack_output_status",
    "waveform_qc_status", "rejection_reason", "final_event_status",
]
if len(nodal_stack_members):
    event_catalog_qc = event_catalog_qc.merge(
        nodal_stack_members[member_qc_columns],
        on="nodal_event_id", how="left", validate="one_to_one",
    )
else:
    for column in member_qc_columns[1:]:
        event_catalog_qc[column] = np.nan

for column in ["waveform_qc_status", "rejection_reason", "final_event_status", "stack_output_status"]:
    event_catalog_qc[column] = event_catalog_qc[column].astype("object")

unassigned_mask = ~event_catalog_qc.get("accepted", False).astype(bool)
not_processed_mask = event_catalog_qc.get("accepted", False).astype(bool) & event_catalog_qc["stack_id"].isna()
event_catalog_qc.loc[unassigned_mask, "waveform_qc_status"] = "not_run_unassigned"
event_catalog_qc.loc[unassigned_mask, "final_event_status"] = "unassigned"
event_catalog_qc.loc[not_processed_mask, "waveform_qc_status"] = "not_run_insufficient_stack_candidates"
event_catalog_qc.loc[not_processed_mask, "final_event_status"] = "not_stacked_insufficient_candidates"
event_catalog_qc["accepted_for_stack"] = event_catalog_qc["accepted_for_stack"].fillna(False).astype(bool)
event_catalog_qc["included_in_output_stack"] = event_catalog_qc["included_in_output_stack"].fillna(False).astype(bool)
assert len(event_catalog_qc) == len(event_catalog_base)
assert event_catalog_qc["nodal_event_id"].is_unique

print("Stacks:", len(nodal_stacks))
print("Members:", len(nodal_stack_members))
print("Reference candidates evaluated:", len(nodal_stack_reference_candidates))
print("Files:", len(nodal_stack_files))
print("Errors:", len(nodal_stack_processing_errors))
print("Enriched event catalog rows:", len(event_catalog_qc))

display(nodal_stacks.head())
display(nodal_stack_processing_errors.head())

[1/194] GEODE_T1_1M_REFRACTION_F3006
[2/194] GEODE_T1_1M_REFRACTION_F3008


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


[3/194] GEODE_T1_1M_REFRACTION_F3009


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


[4/194] GEODE_T1_1M_REFRACTION_F3010


...

[194/194] GEODE_T3_1M_REFRACTION_F4039


/Users/thompsong/miniforge3/envs/flovopy_asl/lib/python3.12/site-packages/obspy/io/mseed/core.py:1085: UserWarning: File will be written with more than one different encodings.
This might have a negative influence on the compatibility with other programs.
  warnings.warn(msg % 'encodings')


Stacks: 194
Members: 961
Reference candidates evaluated: 946
Files: 582
Errors: 0
Enriched event catalog rows: 3436


,stack_id,geode_event_id,geode_survey,line,file_no,source_x_truth_m,source_type,n_candidate_members,n_accepted_members,n_rejected_members,reference_nodal_event_id,reference_supported_other_members,reference_median_corr_to_other_members,median_xcorr_shift_s,median_xcorr_corrcoef,min_member_time_from_final_s,max_member_time_from_final_s,output_dir,status
0,NODALSTACK_T1_T1_1m_refraction_F3006_x0084.5m,GEODE_T1_1M_REFRACTION_F3006,T1_1m_refraction,T1,3006,84.5,hammer,6,6,0,T1_N2_Refraction1m_T1_N2_E00007,5,0.942585,-0.0360,0.945346,-46.990,-0.286,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok
1,NODALSTACK_T1_T1_1m_refraction_F3008_x0088.5m,GEODE_T1_1M_REFRACTION_F3008,T1_1m_refraction,T1,3008,88.5,hammer,3,3,0,T1_N2_Refraction1m_T1_N2_E00011,2,0.957434,0.0060,0.961464,-21.410,-7.324,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok
2,NODALSTACK_T1_T1_1m_refraction_F3009_x0090.5m,GEODE_T1_1M_REFRACTION_F3009,T1_1m_refraction,T1,3009,90.5,hammer,7,7,0,T1_N2_Refraction1m_T1_N2_E00018,6,0.950005,0.0180,0.968298,-42.758,0.060,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok
3,NODALSTACK_T1_T1_1m_refraction_F3010_x0092.5m,GEODE_T1_1M_REFRACTION_F3010,T1_1m_refraction,T1,3010,92.5,hammer,2,2,0,T1_N2_Refraction1m_T1_N2_E00035,1,0.973912,0.0570,0.986956,-43.296,-33.280,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok
4,NODALSTACK_T1_T1_1m_refraction_F3011_x0094.5m,GEODE_T1_1M_REFRACTION_F3011,T1_1m_refraction,T1,3011,94.5,hammer,6,6,0,T1_N2_Refraction1m_T1_N2_E00042,5,0.966665,-0.0085,0.973318,-46.552,-0.332,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,ok


""


## 9. Write notebook-95 tables and CSV exports safely

In [9]:
NOTEBOOK_95_TABLES = [
    "nodal_stacks",
    "nodal_stack_members",
    "nodal_stack_files",
    "nodal_stack_processing_errors",
    "nodal_stack_reference_candidates",
    "nodal_event_catalog_qc",
]

REQUIRED_INPUT_TABLES = [
    "shot_events",
    "shot_gather_files",
    "trace_index",
    "geode_events",
    "nodal_source_estimates",
    "geode_nodal_event_matches",
    "geode_nodal_stack_match_summary",
    "nodal_event_catalog",
]

if nodal_stacks is None or len(nodal_stacks.columns) == 0:
    nodal_stacks = pd.DataFrame(columns=["stack_id", "geode_event_id", "status"])
if nodal_stack_members is None or len(nodal_stack_members.columns) == 0:
    nodal_stack_members = pd.DataFrame(columns=["stack_id", "geode_event_id", "nodal_event_id", "accepted_for_stack"])
if nodal_stack_files is None or len(nodal_stack_files.columns) == 0:
    nodal_stack_files = pd.DataFrame(columns=["stack_id", "geode_event_id", "component", "file_type", "file_path"])
if nodal_stack_processing_errors is None or len(nodal_stack_processing_errors.columns) == 0:
    nodal_stack_processing_errors = pd.DataFrame(columns=["stack_id", "geode_event_id", "stage", "error"])
if nodal_stack_reference_candidates is None or len(nodal_stack_reference_candidates.columns) == 0:
    nodal_stack_reference_candidates = pd.DataFrame(columns=["stack_id", "reference_nodal_event_id", "selected_reference"])
if event_catalog_qc is None or len(event_catalog_qc.columns) == 0:
    event_catalog_qc = pd.DataFrame(columns=["nodal_event_id", "waveform_qc_status", "final_event_status"])

with sqlite3.connect(CATALOG_DB) as conn:
    existing_tables = pd.read_sql(
        """
        SELECT name
        FROM sqlite_master
        WHERE type='table'
        ORDER BY name
        """,
        conn,
    )["name"].tolist()

    missing_inputs = [t for t in REQUIRED_INPUT_TABLES if t not in existing_tables]
    if missing_inputs:
        raise RuntimeError(f"Refusing to write: input tables missing: {missing_inputs}")

    for table_name in NOTEBOOK_95_TABLES:
        conn.execute(f'DROP TABLE IF EXISTS "{table_name}"')

    nodal_stacks.to_sql("nodal_stacks", conn, if_exists="fail", index=False)
    nodal_stack_members.to_sql("nodal_stack_members", conn, if_exists="fail", index=False)
    nodal_stack_files.to_sql("nodal_stack_files", conn, if_exists="fail", index=False)
    nodal_stack_processing_errors.to_sql("nodal_stack_processing_errors", conn, if_exists="fail", index=False)
    nodal_stack_reference_candidates.to_sql("nodal_stack_reference_candidates", conn, if_exists="fail", index=False)
    event_catalog_qc.to_sql("nodal_event_catalog_qc", conn, if_exists="fail", index=False)

    conn.commit()

csv_dir = OUT_ROOT / "catalog_exports"
csv_dir.mkdir(parents=True, exist_ok=True)

def rounded_export(frame):
    out = frame.copy()
    for column in [
        "source_x_truth_m", "estimated_source_x_m", "source_x_residual_m",
        "source_x_m", "adopted_source_x_m", "energy_x_m",
        "nodal_receiver_first_m", "nodal_receiver_last_m",
    ]:
        if column in out.columns:
            out[column] = pd.to_numeric(out[column], errors="coerce").round(POSITION_EXPORT_DECIMALS)
    for column in [
        "xcorr_shift_s", "xcorr_corrcoef", "time_from_final_trigger_s",
        "median_xcorr_shift_s", "median_xcorr_corrcoef",
        "median_corr_to_other_members",
    ]:
        if column in out.columns:
            out[column] = pd.to_numeric(out[column], errors="coerce").round(3)
    return out

rounded_export(nodal_stacks).to_csv(csv_dir / "nodal_stacks.csv", index=False)
rounded_export(nodal_stack_members).to_csv(csv_dir / "nodal_stack_members.csv", index=False)
nodal_stack_files.to_csv(csv_dir / "nodal_stack_files.csv", index=False)
nodal_stack_processing_errors.to_csv(csv_dir / "nodal_stack_processing_errors.csv", index=False)
rounded_export(nodal_stack_reference_candidates).to_csv(csv_dir / "nodal_stack_reference_candidates.csv", index=False)
rounded_export(event_catalog_qc).to_csv(csv_dir / "nodal_event_catalog_qc.csv", index=False)

print("CATALOG_DB:", CATALOG_DB)
print("Dropped/replaced only 95-owned SQLite tables:")
for t in NOTEBOOK_95_TABLES:
    print(" ", t)
print("CSV exports:", csv_dir)

CATALOG_DB: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
Dropped/replaced only 95-owned SQLite tables:
  nodal_stacks
  nodal_stack_members
  nodal_stack_files
  nodal_stack_processing_errors
  nodal_stack_reference_candidates
  nodal_event_catalog_qc
CSV exports: /Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_by_geode/catalog_exports


## 10. Summary QC

In [10]:
if len(nodal_stacks):
    display(
        nodal_stacks
        .groupby(["geode_survey", "line"], dropna=False)
        .agg(
            n_stacks=("stack_id", "count"),
            total_accepted_members=("n_accepted_members", "sum"),
            median_members=("n_accepted_members", "median"),
            median_corr=("median_xcorr_corrcoef", "median"),
        )
        .reset_index()
    )

if len(nodal_stack_files):
    display(
        nodal_stack_files
        .groupby(["component", "file_type"], dropna=False)
        .size()
        .reset_index(name="n_files")
    )

if len(nodal_stack_processing_errors):
    print("Processing errors:")
    display(nodal_stack_processing_errors.head(30))

if len(nodal_stack_members):
    display(
        nodal_stack_members["rejection_reason"]
        .value_counts(dropna=False).rename_axis("reason").reset_index(name="n_events")
    )

display(
    event_catalog_qc["final_event_status"]
    .value_counts(dropna=False).rename_axis("final_event_status").reset_index(name="n_events")
)

,geode_survey,line,n_stacks,total_accepted_members,median_members,median_corr
0,T1_1m_refraction,T1,39,252,6.0,0.971721
1,T1_2m_refraction,T1,36,222,6.0,0.981482
2,T1_streamer_masw,T1,80,294,3.0,0.981374
3,T3_1m_refraction,T3,39,187,5.0,0.964198


,component,file_type,n_files
0,Z,mseed,194
1,Z,png_wiggle,194
2,Z,segy,194


,reason,n_events
0,accepted_waveform,761
1,accepted_consensus_reference,194
2,rejected_low_correlation_probable_nonshot,6


,final_event_status,n_events
0,unassigned,2470
1,accepted_for_stack,955
2,rejected_probable_nonshot_or_line_disturbance,6
3,not_stacked_insufficient_candidates,5
